**File:** notebooks/training_curves.ipynb  
**Owner:** Shamathmika

**Purpose:**  
Visualises training and validation loss curves for DocRes and NAFNet.  
Marks the best checkpoint epoch, plots the learning rate schedule,  
and includes written observations on convergence and overfitting.

**Dependencies:**  
- `checkpoints/docres_loss_log.csv` — produced by `train/train_docres.py`  
- `checkpoints/nafnet_loss_log.csv` — produced by `train/train_nafnet.py`  
- Each CSV must have columns: `epoch`, `train_loss`, `val_loss`, `lr`

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DocRestore'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

DOCRES_LOG = Path('checkpoints/docres_loss_log.csv')
NAFNET_LOG = Path('checkpoints/nafnet_loss_log.csv')

for p in [DOCRES_LOG, NAFNET_LOG]:
    status = 'OK' if p.exists() else 'MISSING — run training first'
    print(f'  [{status}] {p}')

In [ ]:
# Load logs — skip missing files gracefully
logs = {}
for name, path in [('DocRes', DOCRES_LOG), ('NAFNet', NAFNET_LOG)]:
    if path.exists():
        logs[name] = pd.read_csv(path)
        print(f'{name}: {len(logs[name])} epochs logged')
    else:
        print(f'{name}: SKIPPED — {path} not found')

if not logs:
    raise FileNotFoundError('No loss logs found. Run training scripts first.')

In [ ]:
# Plot train and val loss curves for both models on the same axes
fig, ax = plt.subplots(figsize=(10, 5))

colors = {'DocRes': ('#2196F3', '#90CAF9'), 'NAFNet': ('#E53935', '#EF9A9A')}

best_epochs = {}
for model, df in logs.items():
    train_color, val_color = colors[model]
    ax.plot(df['epoch'], df['train_loss'], color=train_color,
            linewidth=2, label=f'{model} train')
    ax.plot(df['epoch'], df['val_loss'],   color=val_color,
            linewidth=2, linestyle='--', label=f'{model} val')

    # Best checkpoint = epoch with lowest val loss
    best_idx = df['val_loss'].idxmin()
    best_epoch = df.loc[best_idx, 'epoch']
    best_val   = df.loc[best_idx, 'val_loss']
    best_epochs[model] = best_epoch

    ax.axvline(best_epoch, color=train_color, linestyle=':', linewidth=1.5,
               label=f'{model} best ckpt (ep {int(best_epoch)})')
    ax.annotate(f'ep {int(best_epoch)}\n{best_val:.4f}',
                xy=(best_epoch, best_val),
                xytext=(best_epoch + 0.5, best_val * 1.05),
                fontsize=8, color=train_color)

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Loss — DocRes vs NAFNet', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for model, ep in best_epochs.items():
    print(f'{model} best checkpoint: epoch {int(ep)}')

In [ ]:
# Plot learning rate schedule over epochs
fig, ax = plt.subplots(figsize=(10, 3))

for model, df in logs.items():
    if 'lr' not in df.columns:
        print(f'{model}: no lr column in log, skipping')
        continue
    train_color, _ = colors[model]
    ax.plot(df['epoch'], df['lr'], color=train_color, linewidth=2, label=model)

ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## TODO: Observations

**Convergence**  
- [ ] Did the models converge smoothly or show instability?
- [ ] How many epochs until the loss plateau?

**Overfitting**  
- [ ] Is there a growing gap between train and val loss?
- [ ] Which model overfit more?

**Best checkpoint**  
- [ ] At what epoch did each model reach its best val loss?
- [ ] Was early stopping warranted?

**Learning rate**  
- [ ] Did cosine annealing behave as expected?
- [ ] Any signs the LR decayed too fast or too slow?